In [14]:
from pathlib import Path

HOME_DIR = Path.home()
FILES_DIR = HOME_DIR / ".files"

MODEL_DIR = FILES_DIR / "models"
INPUT_DIR = FILES_DIR / "input"
OUTPUT_DIR = FILES_DIR / "output"

INPUT_VIDEO_DIR = INPUT_DIR / "videos"
OUTPUT_VIDEO_DIR = OUTPUT_DIR / "videos"

for directory in [
    MODEL_DIR,
    INPUT_DIR,
    OUTPUT_DIR,
    INPUT_VIDEO_DIR,
    OUTPUT_VIDEO_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Directory structure created successfully!")
print("Model directory:", MODEL_DIR)
print("Input video directory:", INPUT_VIDEO_DIR)
print("Output video directory:", OUTPUT_VIDEO_DIR)

Directory structure created successfully!
Model directory: /Users/startup/.files/models
Input video directory: /Users/startup/.files/input/videos
Output video directory: /Users/startup/.files/output/videos


In [15]:
import shutil
from pathlib import Path

SOURCE_VIDEO = Path.home() / "Downloads" / "rally_101047_101584.mp4"

DESTINATION_VIDEO = INPUT_VIDEO_DIR / "input_video.mp4"

if SOURCE_VIDEO.exists():
    shutil.copy2(SOURCE_VIDEO, DESTINATION_VIDEO)
    print("Video copied successfully!")
    print("Destination:", DESTINATION_VIDEO)
else:
    print("Video not found in Downloads!")

Video copied successfully!
Destination: /Users/startup/.files/input/videos/input_video.mp4


In [16]:

MODEL_PATH = MODEL_DIR / "yolo11n.pt"

if MODEL_PATH.exists():
    print("YOLO11n model found!")
    print("Model location:", MODEL_PATH)
else:
    raise FileNotFoundError(
        f"YOLO11n model not found at: {MODEL_PATH}"
    )


YOLO11n model found!
Model location: /Users/startup/.files/models/yolo11n.pt


In [17]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO




In [18]:

INPUT_VIDEO_DIR = INPUT_DIR / "videos"
OUTPUT_VIDEO_DIR = OUTPUT_DIR / "videos"

INPUT_VIDEO_PATH = INPUT_VIDEO_DIR / "input_video.mp4"
OUTPUT_VIDEO_PATH = OUTPUT_VIDEO_DIR / "detected_video.mp4"

OUTPUT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print("Input video path:", INPUT_VIDEO_PATH)
print("Output video path:", OUTPUT_VIDEO_PATH)


Input video path: /Users/startup/.files/input/videos/input_video.mp4
Output video path: /Users/startup/.files/output/videos/detected_video.mp4


In [19]:

model = YOLO(str(MODEL_PATH))

print("YOLO11n model loaded successfully!")


YOLO11n model loaded successfully!


In [20]:

CONFIDENCE_THRESHOLD = 0.25
IMAGE_SIZE = 640
PERSON_CLASS_ID = 0
DEVICE = "cpu"

print("Detection configuration:")
print("Confidence threshold:", CONFIDENCE_THRESHOLD)
print("Image size:", IMAGE_SIZE)
print("Person class ID:", PERSON_CLASS_ID)
print("Device:", DEVICE)


Detection configuration:
Confidence threshold: 0.25
Image size: 640
Person class ID: 0
Device: cpu


In [21]:
if INPUT_VIDEO_PATH.exists():
    print("Input video found!")
    print("Video location:", INPUT_VIDEO_PATH)
else:
    raise FileNotFoundError(
        f"Input video not found: {INPUT_VIDEO_PATH}\n"
        "Place your video inside .files/input/videos "
        "or update INPUT_VIDEO_PATH with the correct filename."
    )

Input video found!
Video location: /Users/startup/.files/input/videos/input_video.mp4


In [22]:
cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("Could not open the input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

duration = total_frames / fps if fps > 0 else 0

print("Video properties:")
print("FPS:", fps)
print("Width:", frame_width)
print("Height:", frame_height)
print("Total frames:", total_frames)
print("Duration (seconds):", round(duration, 2))

cap.release()

Video properties:
FPS: 30.0
Width: 1920
Height: 1080
Total frames: 538
Duration (seconds): 17.93


In [23]:
cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("Could not open the input video.")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO_PATH),
    fourcc,
    fps,
    (frame_width, frame_height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video.")

print("Video capture initialized successfully!")
print("Output video writer initialized successfully!")

Video capture initialized successfully!
Output video writer initialized successfully!


In [24]:
def extract_detections(result):
    detections = []

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        confidence = box.conf[0].item()
        class_id = int(box.cls[0].item())

        width = x2 - x1
        height = y2 - y1

        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        detection = {
            "class_id": class_id,
            "confidence": confidence,
            "x1": int(x1),
            "y1": int(y1),
            "x2": int(x2),
            "y2": int(y2),
            "width": int(width),
            "height": int(height),
            "center_x": int(center_x),
            "center_y": int(center_y)
        }

        detections.append(detection)

    return detections


In [25]:
def draw_detections(frame, detections):
    for detection in detections:
        x1 = detection["x1"]
        y1 = detection["y1"]
        x2 = detection["x2"]
        y2 = detection["y2"]
        confidence = detection["confidence"]

        # Draw bounding box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Add confidence score
        label = f"Person: {confidence:.2f}"

        cv2.putText(
            frame,
            label,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    return frame

In [26]:
frame_count = 0
total_detections = 0

try:
    while True:
        ret, frame = cap.read()

        if not ret:
            break

        # Run YOLO11n detection on the current frame
        results = model.predict(
            source=frame,
            classes=[PERSON_CLASS_ID],
            conf=CONFIDENCE_THRESHOLD,
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            verbose=False
        )

        result = results[0]

        # Extract detection details
        detections = extract_detections(result)
        total_detections += len(detections)

        # Draw bounding boxes
        annotated_frame = draw_detections(frame, detections)

        # Save the annotated frame
        writer.write(annotated_frame)

        frame_count += 1

        if frame_count % 30 == 0:
            print(f"Processed {frame_count} frames...")

finally:
    cap.release()
    writer.release()

print("\nVideo processing completed!")
print("Total frames processed:", frame_count)
print("Total person detections:", total_detections)

Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processed 390 frames...
Processed 420 frames...
Processed 450 frames...
Processed 480 frames...
Processed 510 frames...

Video processing completed!
Total frames processed: 538
Total person detections: 4287


In [27]:
if OUTPUT_VIDEO_PATH.exists():
    file_size_mb = OUTPUT_VIDEO_PATH.stat().st_size / (1024 * 1024)

    print("Output video saved successfully!")
    print("Output location:", OUTPUT_VIDEO_PATH)
    print(f"File size: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError(
        f"Output video not found: {OUTPUT_VIDEO_PATH}"
    )

Output video saved successfully!
Output location: /Users/startup/.files/output/videos/detected_video.mp4
File size: 23.60 MB


In [28]:
from IPython.display import Video, display

display(Video(
    str(OUTPUT_VIDEO_PATH),
    embed=True,
    width=700
))